<a href="https://colab.research.google.com/github/PauloRadatz/py_dss_toolkit/blob/master/examples/py-dss-toolkit_tutorial/9-Visualize_Feeder_Topology_numerical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Circuit Interactive View (Numerical)

This notebook demonstrates how to visualize a distribution feeder circuit using the `py_dss_toolkit` package. We’ll compile an OpenDSS model, solve a power flow, and generate interactive circuit plots for numerical variables—while exploring practical customization options.

**Contact:** paulo.radatz@gmail.com

If you’d like a structured learning path:
- **OpenDSS courses:** https://www.pauloradatz.me/opendss-courses
- **Learn the basics of controlling OpenDSS via Python (py-dss-interface course):** https://www.pauloradatz.me/course-py-dss-interface

## Install packages

We’ll install `py-dss-toolkit`. During installation, `pip` will also install `py-dss-interface`, which is a required dependency.

- **`py-dss-toolkit`**: utilities to run OpenDSS studies and generate high-level visualizations (including interactive circuit plots).
- **`py-dss-interface`**: a Python package that controls **OpenDSS Powered by EPRI** directly from Python.

In [1]:
!pip install py-dss-toolkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.9/119.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 41.1 MB/s eta 0:00:00


## Download the example feeder

Next, we’ll clone the `opendss-python-examples` repository, which includes ready-to-run OpenDSS feeder models.  
We’ll use one of these feeders as the input circuit for the visualization examples in this notebook.

In [2]:
!git clone https://github.com/PauloRadatz/opendss-python-examples

Cloning into 'opendss-python-examples'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 31 (delta 3), reused 31 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 176.48 KiB | 2.85 MiB/s, done.
Resolving deltas: 100% (3/3), done.


## Define the DSS master file path

Now we set the path to the **OpenDSS master file** (`Master_ckt5.dss`).  
This file is the entry point of the feeder model—it typically redirects to all the other DSS files (lines, loads, transformers, etc.) needed to compile the circuit.

In [3]:
dss_file_path = "/content/opendss-python-examples/feeder_models/EPRITestCircuits/ckt5/Master_ckt5.dss"

## Imports

We’ll use `py-dss-interface` together with `py-dss-toolkit`:

- **`py_dss_interface`** provides a direct connection to the OpenDSS engine.
- **`dss_tools`** (from `py_dss_toolkit`) builds on top of that connection to simplify common workflows—such as accessing models, retrieving results, and creating interactive visualizations.

In [4]:
import py_dss_interface
from py_dss_toolkit import dss_tools

## Initialize OpenDSS and connect it to `dss_tools`

Next, we create a DSS engine instance using `py_dss_interface` to initialize the simulation environment.

Then we connect that same DSS instance to `dss_tools`, so all toolkit helper functions operate on the exact engine we just created.

In [5]:
dss = py_dss_interface.DSS()
dss_tools.update_dss(dss)
dss.started

True

## Compile the model and solve a power flow

Now we compile the OpenDSS master file and run a power flow solution:

- **`compile`** loads the circuit and all referenced files (via `Redirect` commands). For more details, see **Module 2** of: https://www.pauloradatz.me/course-snapshot  
- **`solve`** runs the power flow so voltages, currents, and power flows are available for visualization and analysis.

In [6]:
dss.text(f"compile [{dss_file_path}]")
dss.text(f"solve")

''

## Plot active power in the circuit

Now we’ll create an interactive circuit plot using **active power** as the displayed parameter.

We’ll start with the default settings, then later customize the figure (for example: title, line width, and other visualization options).


In [7]:
# Plot active power using default parameters
dss_tools.interactive_view.circuit_plot(parameter="active power")

## Two ways to customize `circuit_plot()`

There are two main ways to change how the interactive circuit plot looks:

1. **Method parameters (most common)**  
   Pass options directly into `circuit_plot(...)`—like `title`, `width_2ph`, `width_1ph`, and others.  
   This is the approach you’ll use most of the time, and it’s why the most common settings are exposed as method parameters.

2. **Settings object**  
   For additional options that are not exposed as parameters, you can edit the settings object associated with the plot type.  
   For active power, that object is: `dss_tools.interactive_view.active_power_settings`  

In the next cells, we’ll use both approaches so you can recognize when each one is more convenient.

In [8]:
# Customize plot: Active power with title and custom line widths
dss_tools.interactive_view.circuit_plot(parameter="active power", title="Active Power [KW]", width_2ph=2, width_1ph=1)

In [9]:
# Adjust power settings and plot active power with modified settings
dss_tools.interactive_view.active_power_settings.colorbar_cmax = 5000
dss_tools.interactive_view.active_power_settings.colorbar_cmin = -1000
dss_tools.interactive_view.active_power_settings.colorbar_title = "P max = 5000 kW"
dss_tools.interactive_view.circuit_plot(parameter="active power", title="Active Power [KW] with changes in the power settings")

## Plot other numerical variables (example: voltage)

The circuit plot isn’t limited to power—we can plot other numerical parameters, such as **voltage**.

A key difference compared to active power:

- **Active power** is typically a *single value per line segment* (total kW flow), so coloring the circuit is straightforward.
- **Voltage** in a 3-phase system can produce *multiple values per segment* (one per node/phase).

Since the plot can only color each segment with **one** value, we must choose how to reduce the multi-phase voltage into a single number. In `py-dss-toolkit`, you can select:
- `"min"`: colors by the lowest phase voltage
- `"max"`: colors by the highest phase voltage
- `"mean"`: colors by the average phase voltage

In the next cell, we’ll configure the voltage settings (color scale limits and reduction rule) and then plot the voltage profile.

In [10]:
# Plot the voltage in the circuit
dss_tools.interactive_view.voltage_settings.colorbar_cmax = 1.1
dss_tools.interactive_view.voltage_settings.colorbar_cmin = 0.9
dss_tools.interactive_view.voltage_settings.nodes_voltage_value = "min"
dss_tools.interactive_view.circuit_plot(parameter="voltage", title="Voltage [pu]")

## Mark buses on the circuit plot

In addition to coloring the circuit by a parameter (like active power), we can also **highlight specific buses** with markers.

This is useful when you want to:
- Call attention to a bus of interest (e.g., a critical customer, a regulator location, a voltage issue)
- Compare results around a specific area of the feeder
- Guide the viewer during a presentation or troubleshooting workflow

In [11]:
# Mark a specific bus in the circuit plot
bus_list = [dss_tools.interactive_view.circuit_get_bus_marker(name="51012", marker_name="My Bus", color="red", size=20)]
dss_tools.interactive_view.circuit_plot(parameter="active power", title="Active Power [KW] with Marked Bus", bus_markers=bus_list)

## Wrap-up

You’ve now seen a complete workflow to visualize an OpenDSS feeder with `py-dss-toolkit`:

- Install the packages and load an example feeder model
- Compile and solve a power flow with OpenDSS
- Create interactive circuit plots (active power, voltage, etc.)
- Customize plots using numerical variables **method parameters** and **settings objects**
- Add **bus markers** to highlight locations of interest

If you have questions, suggestions, or ideas for additional features/examples, feel free to reach out:

**Contact:** paulo.radatz@gmail.com

More learning resources:
- **OpenDSS courses:** https://www.pauloradatz.me/opendss-courses  
- **Python + OpenDSS fundamentals (py-dss-interface course):** https://www.pauloradatz.me/course-py-dss-interface